In [4]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 1. Load the data
data = xr.open_dataset("../data/currents-2020.nc")

# Keep only the surface currents
data = data.isel(depth=0)

# Select the study area
data = data.sel(longitude=slice(-100, 20),
                latitude=slice(-60, 70))

# Reduce the number of points to make the animation faster
data = data.coarsen(longitude=4, latitude=4, boundary="trim").mean().load()
# 2. Calculate current speed
speed = np.sqrt(data.uo**2 + data.vo**2)

# Calculate the average speed over the whole year
yearly_average = speed.mean(dim="time")

# Calculate the correlation for each month
correlation = xr.corr(speed, yearly_average,
                      dim=["latitude", "longitude"])

# Get the month names
months = [str(date)[:7] for date in data.time.values]


# 3. Create the two maps
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# First map: monthly current speed
map1 = ax1.pcolormesh(data.longitude, data.latitude, speed[0],
                      vmin=0, vmax=0.5, cmap="viridis")

fig.colorbar(map1, ax=ax1, label="Current speed (m/s)")

# Add arrows to show current direction
step = 6

arrows = ax1.quiver(
    data.longitude[::step], data.latitude[::step],
    data.uo[0, ::step, ::step],
    data.vo[0, ::step, ::step],
    color="white"
)

# Second map: yearly average current speed
map2 = ax2.pcolormesh(data.longitude, data.latitude, yearly_average,
                      vmin=0, vmax=0.5, cmap="viridis")

fig.colorbar(map2, ax=ax2, label="Average speed (m/s)")

ax1.set_title("Monthly current speed and direction")
ax2.set_title("Yearly average current speed")


# 4. Create the animation
def update(month):

    # Update the first map
    map1.set_array(speed[month].values.ravel())

    # Update the direction arrows
    arrows.set_UVC(
        data.uo[month, ::step, ::step],
        data.vo[month, ::step, ::step]
    )

    # Update the month and correlation
    fig.suptitle(
        f"{months[month]} | Correlation: {correlation[month].values:.2f}"
    )

    return map1, arrows


animation = FuncAnimation(
    fig,
    update,
    frames=len(data.time),
    interval=800
)

plt.close()

HTML(animation.to_jshtml())